In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE = "/content/drive/MyDrive/anon/pidgindataset2026October/"

In [4]:
import pandas as pd

file_path = BASE + "root/Metadata.csv"

df = pd.read_csv(file_path)

In [5]:
df.head()

,speaker_id,audio_path,pidgin_transcription,english_translation,medical_terms_english,medical_terms_pidgen,audio_length,sample_rate,num_channels,speech_rate,signal_to_noise_ratio,silence_ratio,audio,file_exists,text,reference,sample_id
0,Speaker_1,root/audio/Speaker_1/Speaker1-1.wav,e depend on your effort how you take dey do ex...,it all depends on your efforts regarding regul...,"['care', 'knee', 'walk', 'two days', 'one to t...","['exercise', 'everytime', 'after two day', 'no...",19.04,16000.0,1.0,2.678378,54.947548,0.319377,/content/drive/MyDrive/Brock_University/pidgin...,True,e depend on your effort how you take dey do ex...,e depend on your effort how you take dey do ex...,6a840a4ad1fd60283227
1,Speaker_1,root/audio/Speaker_1/Speaker1-2.wav,my daughter make four years two nights ago she...,my daughter is four years old two nights ago s...,"['daughter', 'four years old', 'vomited', 'sor...","['daughter', 'wake up', 'vomited', 'pain', 'ho...",34.48,16000.0,1.0,2.580879,53.953575,0.334655,/content/drive/MyDrive/Brock_University/pidgin...,True,my daughter make four years two nights ago she...,my daughter make four years two nights ago she...,fa777d58c8f323da62e2
2,Speaker_1,root/audio/Speaker_1/Speaker1-3.wav,hi dis na retention cyst e go need excise for ...,hi this is definitely a retention cyst it need...,"['retention cyst', 'mass', 'histopathological ...","['reasons', 'removed', 'important part of the ...",21.97,16000.0,1.0,2.321869,57.647476,0.335672,/content/drive/MyDrive/Brock_University/pidgin...,True,hi dis na retention cyst e go need excise for ...,hi dis na retention cyst e go need excise for ...,01fceaa9e74b77698df4
3,Speaker_1,root/audio/Speaker_1/Speaker1-4.wav,i be thirty one years woman and i get blod clo...,i am a thirty one year old woman with a blood ...,"['thirty one year old woman', 'head injury', '...","['thirty one years woman', 'not', 'head injury...",34.61,16000.0,1.0,2.571562,49.890247,0.308394,/content/drive/MyDrive/Brock_University/pidgin...,True,i be thirty one years woman and i get blod clo...,i be thirty one years woman and i get blod clo...,3d812380dee30f23b366
4,Speaker_1,root/audio/Speaker_1/Speaker1-5.wav,i be forty five years old man my platelet coun...,i am forty five years old male my platelet cou...,"['forty five years old male', 'platelet count'...","['forty five years old man', 'platelet count',...",42.63,16000.0,1.0,2.040783,50.834831,0.307166,/content/drive/MyDrive/Brock_University/pidgin...,True,i be forty five years old man my platelet coun...,i be forty five years old man my platelet coun...,ac3bdc92665cb5600166


In [6]:
df.describe()

,audio_length,sample_rate,num_channels,speech_rate,signal_to_noise_ratio,silence_ratio
count,10218.000000,6466.0,6466.0,6466.000000,6466.000000,6466.000000
mean,27.826377,16000.0,1.0,2.410646,33.613259,0.186540
std,12.653020,0.0,0.0,0.502543,11.073948,0.097676
min,2.150000,16000.0,1.0,0.132050,5.822312,0.000000
25%,19.310000,16000.0,1.0,2.083962,26.374675,0.128555
50%,25.310000,16000.0,1.0,2.402571,32.541592,0.187781
75%,34.025000,16000.0,1.0,2.734375,38.315322,0.256530
max,124.390000,16000.0,1.0,14.694917,65.283104,0.717644


In [7]:
audio_path = BASE + df.audio_path.get(0)

In [8]:
from IPython.display import Audio

Audio(audio_path)

In [9]:
!pip install transformers torch pandas -q

In [ ]:
from transformers import pipeline
import pandas as pd

# NER with offsets + grouping
ner = pipeline(
    "ner",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple",  # groups subwords
    device=0  # or -1 for CPU
)

config.json:   0%|          | 0.00/5.00k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  266MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
import pandas as pd

def extract_terms_from_text_merge(text, threshold=0.7, max_gap=1):
    if pd.isna(text) or not str(text).strip():
        return []

    text = str(text)
    ents = ner(text)

    spans = []
    for e in ents:
        if e.get("score", 0) >= threshold:
            spans.append({
                "start": e["start"],
                "end": e["end"],
                "label": e["entity_group"],
                "score": float(e["score"])
            })

    spans = sorted(spans, key=lambda x: x["start"])

    merged = []
    for s in spans:
        if not merged:
            merged.append(s)
            continue

        prev = merged[-1]

        gap_text = text[prev["end"]:s["start"]]

        # merge if spans are touching or separated only by tiny whitespace
        if s["start"] - prev["end"] <= max_gap and gap_text.strip() == "":
            prev["end"] = s["end"]
            prev["score"] = min(prev["score"], s["score"])
        else:
            merged.append(s)

    terms = []
    seen = set()

    for m in merged:
        term = text[m["start"]:m["end"]].strip().lower()

        if term and term not in seen:
            seen.add(term)
            terms.append(term)

    return terms

df["medical_terms_english"] = df["english_translation"].apply(
    lambda x: extract_terms_from_text_merge(x, threshold=0.9)
)

df["medical_terms_pidgen"] = df["pidgin_transcription"].apply(
    lambda x: extract_terms_from_text_merge(x, threshold=0.9)
)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
df.head()

,speaker_id,audio_path,pidgin_transcription,english_translation,medical_terms_english,medical_terms_pidgen
0,Speaker_1,root/audio/Speaker_1/Speaker1-1.wav,e depend on your effort how you take dey do ex...,it all depends on your efforts regarding regul...,"[care, knee, walk, two days, one to two weeks]","[ex, everytime, after two day, normal, one to ..."
1,Speaker_1,root/audio/Speaker_1/Speaker1-2.wav,my daughter make four years two nights ago she...,my daughter is four years old two nights ago s...,"[daughter, four years old, vomit, sore stomach...","[daughter, wake up, vomit, pain, hot, parace, ..."
2,Speaker_1,root/audio/Speaker_1/Speaker1-3.wav,hi dis na retention cyst e go need excise for ...,hi this is definitely a retention cyst it need...,"[retention cyst, mass, histopathological exami...","[reason, remove, important part of the body, e..."
3,Speaker_1,root/audio/Speaker_1/Speaker1-4.wav,i be thirty one years woman and i get blod clo...,i am a thirty one year old woman with a blood ...,"[thirty one year old woman, head injury, ribs,...","[thirty one years woman, no, head injury, inju..."
4,Speaker_1,root/audio/Speaker_1/Speaker1-5.wav,i be forty five years old man my platelet coun...,i am forty five years old male my platelet cou...,"[forty five years old male, platelet count, th...","[forty five years old man, platelet count, thi..."


In [ ]:
null_rows = df[df["medical_terms_english"].isna()]

print("Number of null rows:", len(null_rows))
null_rows.head()

Number of null rows: 0


,speaker_id,audio_path,pidgin_transcription,english_translation,medical_terms_english,medical_terms_pidgen


In [ ]:
import re
import ast
import pandas as pd

def normalize_terms_with_text(row, version_text):
    text = str(row["english_translation"]).lower()
    terms = row[version_text]

    # Convert string representation of list into Python list
    if isinstance(terms, str):
        try:
            terms = ast.literal_eval(terms)
        except (ValueError, SyntaxError):
            return []

    if not isinstance(terms, list):
        return []

    # Extract words from original English sentence
    words = re.findall(r"\b[a-zA-Z]+\b", text)

    cleaned_terms = []
    seen = set()

    for term in terms:
        term = str(term).lower().strip()

        if not term:
            continue

        matched = False

        # Case 1: term is part of a full word
        for word in words:
            if term in word and term != word:
                if word not in seen:
                    cleaned_terms.append(word)
                    seen.add(word)
                matched = True
                break

        # Case 2: keep original term if no better match found
        if not matched:
            if term not in seen:
                cleaned_terms.append(term)
                seen.add(term)

    return cleaned_terms


# Apply normalization to both columns
df["medical_terms_english"] = df.apply(
    normalize_terms_with_text,
    axis=1,
    args=("medical_terms_english",)
)

df["medical_terms_pidgen"] = df.apply(
    normalize_terms_with_text,
    axis=1,
    args=("medical_terms_pidgen",)
)

# Display results
print(df[[
    "english_translation",
    "medical_terms_english",
    "medical_terms_pidgen"
]].head())

                                 english_translation  \
0  it all depends on your efforts regarding regul...   
1  my daughter is four years old two nights ago s...   
2  hi this is definitely a retention cyst it need...   
3  i am a thirty one year old woman with a blood ...   
4  i am forty five years old male my platelet cou...   

                               medical_terms_english  \
0     [care, knee, walk, two days, one to two weeks]   
1  [daughter, four years old, vomited, sore stoma...   
2  [retention cyst, mass, histopathological exami...   
3  [thirty one year old woman, head injury, ribs,...   
4  [forty five years old male, platelet count, th...   

                                medical_terms_pidgen  
0  [exercise, everytime, after two day, normally,...  
1  [daughter, wake up, vomited, pain, hot, parace...  
2  [reasons, removed, important part of the body,...  
3  [thirty one years woman, not, head injury, inj...  
4  [forty five years old man, platelet count, thi..

In [ ]:
df.columns = df.columns.str.lower()

In [ ]:
df.head()

,speaker_id,audio_path,pidgin_transcription,english_translation,medical_terms_english,medical_terms_pidgen
0,Speaker_1,root/audio/Speaker_1/Speaker1-1.wav,e depend on your effort how you take dey do ex...,it all depends on your efforts regarding regul...,"[care, knee, walk, two days, one to two weeks]","[exercise, everytime, after two day, normally,..."
1,Speaker_1,root/audio/Speaker_1/Speaker1-2.wav,my daughter make four years two nights ago she...,my daughter is four years old two nights ago s...,"[daughter, four years old, vomited, sore stoma...","[daughter, wake up, vomited, pain, hot, parace..."
2,Speaker_1,root/audio/Speaker_1/Speaker1-3.wav,hi dis na retention cyst e go need excise for ...,hi this is definitely a retention cyst it need...,"[retention cyst, mass, histopathological exami...","[reasons, removed, important part of the body,..."
3,Speaker_1,root/audio/Speaker_1/Speaker1-4.wav,i be thirty one years woman and i get blod clo...,i am a thirty one year old woman with a blood ...,"[thirty one year old woman, head injury, ribs,...","[thirty one years woman, not, head injury, inj..."
4,Speaker_1,root/audio/Speaker_1/Speaker1-5.wav,i be forty five years old man my platelet coun...,i am forty five years old male my platelet cou...,"[forty five years old male, platelet count, th...","[forty five years old man, platelet count, thi..."


In [ ]:
import soundfile as sf
import os

def get_audio_length_fast(path):
    try:
        info = sf.info(path)
        return info.frames / info.samplerate
    except:
        return None

df["audio_length"] = df["audio_path"].apply(
    lambda x: get_audio_length_fast(os.path.join(BASE, x))
)

In [ ]:
df["audio_length"] = df["audio_length"].round(2)

In [ ]:
df.head()

,speaker_id,audio_path,pidgin_transcription,english_translation,medical_terms_english,medical_terms_pidgen,audio_length
0,Speaker_1,root/audio/Speaker_1/Speaker1-1.wav,e depend on your effort how you take dey do ex...,it all depends on your efforts regarding regul...,"[care, knee, walk, two days, one to two weeks]","[exercise, everytime, after two day, normally,...",19.04
1,Speaker_1,root/audio/Speaker_1/Speaker1-2.wav,my daughter make four years two nights ago she...,my daughter is four years old two nights ago s...,"[daughter, four years old, vomited, sore stoma...","[daughter, wake up, vomited, pain, hot, parace...",34.48
2,Speaker_1,root/audio/Speaker_1/Speaker1-3.wav,hi dis na retention cyst e go need excise for ...,hi this is definitely a retention cyst it need...,"[retention cyst, mass, histopathological exami...","[reasons, removed, important part of the body,...",21.97
3,Speaker_1,root/audio/Speaker_1/Speaker1-4.wav,i be thirty one years woman and i get blod clo...,i am a thirty one year old woman with a blood ...,"[thirty one year old woman, head injury, ribs,...","[thirty one years woman, not, head injury, inj...",34.61
4,Speaker_1,root/audio/Speaker_1/Speaker1-5.wav,i be forty five years old man my platelet coun...,i am forty five years old male my platelet cou...,"[forty five years old male, platelet count, th...","[forty five years old man, platelet count, thi...",42.63


In [10]:
!pip install librosa soundfile numpy pandas -q

In [11]:
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf

def extract_audio_features(rel_path, text):
    path = os.path.join(BASE, rel_path)

    try:
        # --- Basic I/O ---
        y, sr = librosa.load(path, sr=None, mono=False)
        info = sf.info(path)
        num_channels = info.channels

        # ensure mono for analysis
        if y.ndim > 1:
            y_mono = np.mean(y, axis=0)
        else:
            y_mono = y

        duration = len(y_mono) / sr if sr > 0 else 0.0

        # --- Speech rate (words/sec) ---
        if isinstance(text, str) and duration > 0:
            word_count = len(text.strip().split())
            speech_rate = word_count / duration
        else:
            speech_rate = None

        # --- Silence ratio ---
        intervals = librosa.effects.split(y_mono, top_db=30)
        speech_samples = sum((end - start) for start, end in intervals)
        silence_ratio = 1 - (speech_samples / len(y_mono)) if len(y_mono) > 0 else None

        # --- Approx SNR ---
        rms = librosa.feature.rms(y=y_mono)[0]
        signal_power = np.mean(rms**2)

        # estimate noise from quietest 10% frames
        noise_frames = np.percentile(rms, 10)
        noise_power = np.mean((rms[rms <= noise_frames])**2) if np.any(rms <= noise_frames) else 1e-10

        snr = 10 * np.log10(signal_power / (noise_power + 1e-10))

        return pd.Series({
            "sample_rate": sr,
            "num_channels": num_channels,
            "speech_rate": speech_rate,
            "signal_to_noise_ratio": snr,
            "silence_ratio": silence_ratio
        })

    except Exception as e:
        return pd.Series({
            "sample_rate": None,
            "num_channels": None,
            "speech_rate": None,
            "signal_to_noise_ratio": None,
            "silence_ratio": None
        })

In [13]:
feature_cols = [
    "speech_rate",
    "signal_to_noise_ratio",
    "silence_ratio",
    "sample_rate",
    "num_channels"
]

# Create columns if they don't exist
for col in feature_cols:
    if col not in df.columns:
        df[col] = np.nan

# Identify rows with missing features
missing_mask = df[feature_cols].isna().any(axis=1)

print(f"Total rows: {len(df)}")
print(f"Rows requiring extraction: {missing_mask.sum()}")

# Extract features only for incomplete rows
if missing_mask.any():

    df_features = df.loc[missing_mask].apply(
        lambda row: pd.Series(
            extract_audio_features(
                row["audio_path"],
                row["pidgin_transcription"]
            )
        ),
        axis=1
    )

    # Fill missing values without overwriting existing ones
    df[feature_cols] = df[feature_cols].combine_first(
        df_features[feature_cols]
    )

print("Feature extraction completed.")

Total rows: 10218
Rows requiring extraction: 3752
Feature extraction completed.


In [ ]:
df_features = df.apply(
    lambda row: extract_audio_features(row["audio_path"], row["pidgin_transcription"]),
    axis=1
)

df = pd.concat([df, df_features], axis=1)

/tmp/ipykernel_370/1582617669.py:12: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=False)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [15]:
df.tail()

,speaker_id,audio_path,pidgin_transcription,english_translation,medical_terms_english,medical_terms_pidgen,audio_length,sample_rate,num_channels,speech_rate,signal_to_noise_ratio,silence_ratio,audio,file_exists,text,reference,sample_id
10213,Speaker_21,root/audio/Speaker_21/Speaker21-496.wav,"How you dey, I dey experience sharp electric s...","Hi, I am experiencing sharp electric shock pai...","['sharp electric shock pain', 'right elbow', '...","['sharp', 'pain', 'right elbow', 'shoulder sur...",21.90,16000.0,1.0,2.374816,36.192474,0.216676,/content/drive/MyDrive/Brock_University/pidgin...,True,"How you dey, I dey experience sharp electric s...",how you dey i dey experience sharp electric sh...,6440fdf2eb0043ac4572
10214,Speaker_21,root/audio/Speaker_21/Speaker21-497.wav,I get my period on Nov 13th and e last six day...,I got my period on November 13th and it lasted...,"['fiancé', 'unprotected', 'got']","['fianc', 'ovulation', 'positive']",52.29,16000.0,1.0,2.237463,34.063419,0.279664,/content/drive/MyDrive/Brock_University/pidgin...,True,I get my period on Nov 13th and e last six day...,i get my period on nov th and e last six days ...,dc37503fc04764eeea8f
10215,Speaker_21,root/audio/Speaker_21/Speaker21-498.wav,"For di past month, I don dey experience chroni...","For the past month, I have been experiencing c...","['chronic pain', 'left', 'bone in the', 'two w...","['chronic pain', 'di bone', 'hip', 'two weeks ...",42.89,16000.0,1.0,2.051892,33.353992,0.301611,/content/drive/MyDrive/Brock_University/pidgin...,True,"For di past month, I don dey experience chroni...",for di past month i don dey experience chronic...,f208d816db7e3b6020b8
10216,Speaker_21,root/audio/Speaker_21/Speaker21-499.wav,"How you dey, yesterday my girlfriend and I do ...","Hello, yesterday my girlfriend and I had sexua...",[],[],22.55,16000.0,1.0,2.261986,27.396887,0.281018,/content/drive/MyDrive/Brock_University/pidgin...,True,"How you dey, yesterday my girlfriend and I do ...",how you dey yesterday my girlfriend and i do s...,fec1790d96781af736ed
10217,Speaker_21,root/audio/Speaker_21/Speaker21-500.wav,My dad dey currently experience chronic pain f...,My dad is currently experiencing chronic pain ...,"['chronic pain', 'shoulders', 'stiffness', 'si...","['dad', 'chronic pain', 'shoulders', 'stiffnes...",36.90,16000.0,1.0,1.978507,26.493458,0.246311,/content/drive/MyDrive/Brock_University/pidgin...,True,My dad dey currently experience chronic pain f...,my dad dey currently experience chronic pain f...,77b51ff2a591c171ad66


In [16]:
df.to_csv(BASE+"Metadata1.csv", index=False)

In [17]:
# Calculate total duration in seconds
total_seconds = pd.to_numeric(df["audio_length"], errors="coerce").sum()

# Convert to hours, minutes, seconds
hours, remainder = divmod(int(total_seconds), 3600)
minutes, seconds = divmod(remainder, 60)

print(f"Total Duration: {hours} hours, {minutes} minutes, {seconds} seconds")
print(f"Total Hours: {total_seconds / 3600:.2f}")

Total Duration: 78 hours, 58 minutes, 49 seconds
Total Hours: 78.98


In [18]:
df.columns

Index(['speaker_id', 'audio_path', 'pidgin_transcription',
       'english_translation', 'medical_terms_english', 'medical_terms_pidgen',
       'audio_length', 'sample_rate', 'num_channels', 'speech_rate',
       'signal_to_noise_ratio', 'silence_ratio', 'audio', 'file_exists',
       'text', 'reference', 'sample_id'],
      dtype='object')

In [19]:
import pandas as pd
import numpy as np

# General dataset information
print("Dataset Shape:", df.shape)
print("Total Records:", len(df))
print("Total Columns:", len(df.columns))

# Function to safely count unique values
def safe_nunique(series):
    return series.apply(
        lambda x: tuple(x) if isinstance(x, list) else x
    ).nunique(dropna=True)

# Column-level statistics
column_stats = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Non-Null": df.count(),
    "Missing": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(2),
    "Unique Values": df.apply(safe_nunique)
})

display(column_stats)

# Descriptive statistics for numerical columns
display(df.describe(include=[np.number]).T)

Dataset Shape: (10218, 17)
Total Records: 10218
Total Columns: 17


,Data Type,Non-Null,Missing,Missing (%),Unique Values
speaker_id,object,10218,0,0.0,21
audio_path,object,10218,0,0.0,10218
pidgin_transcription,object,10218,0,0.0,9875
english_translation,object,10218,0,0.0,9651
medical_terms_english,object,10218,0,0.0,9250
medical_terms_pidgen,object,10218,0,0.0,9461
audio_length,float64,10218,0,0.0,3493
sample_rate,float64,10218,0,0.0,1
num_channels,float64,10218,0,0.0,1
speech_rate,float64,10218,0,0.0,9737


,count,mean,std,min,25%,50%,75%,max
audio_length,10218.0,27.826377,12.653020,2.150000,19.310000,25.310000,34.025000,124.390000
sample_rate,10218.0,16000.000000,0.000000,16000.000000,16000.000000,16000.000000,16000.000000,16000.000000
num_channels,10218.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
speech_rate,10218.0,2.363408,0.492410,0.132050,2.029071,2.346576,2.685537,14.694917
signal_to_noise_ratio,10218.0,32.039383,10.505199,5.822312,25.182662,30.804038,37.089207,65.283104
silence_ratio,10218.0,0.187849,0.111876,0.000000,0.120457,0.187444,0.259174,0.971912


In [20]:
import pandas as pd
import numpy as np
import ast
from collections import Counter

# Convert numeric columns
numeric_cols = [
    "audio_length",
    "sample_rate",
    "num_channels",
    "speech_rate",
    "signal_to_noise_ratio",
    "silence_ratio"
]

data = df.copy()

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Convert medical-term strings to lists
def parse_terms(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

for col in ["medical_terms_english", "medical_terms_pidgen"]:
    data[col] = data[col].apply(parse_terms)


# -------------------------------
# A. Dataset overview
# -------------------------------
print("=== DATASET OVERVIEW ===")
print("Total samples:", len(data))
print("Total speakers:", data["speaker_id"].nunique())
print("Unique audio paths:", data["audio_path"].nunique())
print("Duplicate audio paths:", data["audio_path"].duplicated().sum())


# -------------------------------
# B. Speaker statistics
# -------------------------------
print("\n=== SPEAKER STATISTICS ===")

speaker_stats = data.groupby("speaker_id").agg(
    utterances=("audio_path", "count"),
    total_duration_seconds=("audio_length", "sum"),
    mean_duration_seconds=("audio_length", "mean"),
    mean_speech_rate=("speech_rate", "mean"),
    mean_snr=("signal_to_noise_ratio", "mean"),
    mean_silence_ratio=("silence_ratio", "mean")
)

speaker_stats["total_duration_hours"] = (
    speaker_stats["total_duration_seconds"] / 3600
)

display(speaker_stats.round(3))


# -------------------------------
# C. Text statistics
# -------------------------------
print("\n=== TEXT STATISTICS ===")

for col in ["pidgin_transcription", "english_translation"]:
    text = data[col].fillna("").astype(str)

    word_counts = text.str.split().str.len()
    char_counts = text.str.len()

    print(f"\nColumn: {col}")
    print("Total words:", word_counts.sum())
    print("Unique words:", len(set(
        word.lower()
        for sentence in text
        for word in sentence.split()
    )))
    print("Average words:", round(word_counts.mean(), 2))
    print("Median words:", word_counts.median())
    print("Min words:", word_counts.min())
    print("Max words:", word_counts.max())
    print("Average characters:", round(char_counts.mean(), 2))
    print("Missing:", data[col].isna().sum())


# -------------------------------
# D. Medical terminology
# -------------------------------
print("\n=== MEDICAL TERM STATISTICS ===")

for col in ["medical_terms_english", "medical_terms_pidgen"]:
    terms = data[col].explode().dropna().astype(str).str.lower()

    frequencies = Counter(terms)

    print(f"\nColumn: {col}")
    print("Total term occurrences:", len(terms))
    print("Unique medical terms:", len(frequencies))
    print(
        "Utterances containing terms:",
        data[col].apply(lambda x: len(x) > 0).sum()
    )
    print(
        "Average terms per utterance:",
        round(data[col].apply(len).mean(), 2)
    )
    print("Top 20 medical terms:")

    display(pd.DataFrame(
        frequencies.most_common(20),
        columns=["Medical Term", "Frequency"]
    ))


# -------------------------------
# E. Audio statistics
# -------------------------------
print("\n=== AUDIO STATISTICS ===")

display(data[numeric_cols].describe().T.round(3))

total_seconds = data["audio_length"].sum()

hours, remainder = divmod(int(total_seconds), 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\nTotal Audio Duration: {hours}h {minutes}m {seconds}s")
print(f"Total Hours: {total_seconds / 3600:.2f}")

print("\nSample Rate Distribution:")
display(data["sample_rate"].value_counts(dropna=False))

print("\nChannel Distribution:")
display(data["num_channels"].value_counts(dropna=False))

=== DATASET OVERVIEW ===
Total samples: 10218
Total speakers: 21
Unique audio paths: 10218
Duplicate audio paths: 0

=== SPEAKER STATISTICS ===


,utterances,total_duration_seconds,mean_duration_seconds,mean_speech_rate,mean_snr,mean_silence_ratio,total_duration_hours
speaker_id,,,,,,,
Speaker_1,499,16083.72,32.232,2.328,51.654,0.301,4.468
Speaker_10,496,12976.39,26.162,2.612,32.513,0.173,3.605
Speaker_11,500,9310.74,18.621,1.966,26.718,0.147,2.586
Speaker_12,501,15949.52,31.835,2.186,34.424,0.227,4.430
Speaker_13,495,13603.06,27.481,2.532,31.437,0.203,3.779
Speaker_14,500,13846.07,27.692,2.373,26.522,0.168,3.846
Speaker_15,500,15939.33,31.879,2.107,33.642,0.239,4.428
Speaker_16,500,13972.24,27.944,2.491,30.492,0.224,3.881
Speaker_17,251,8154.85,32.489,2.188,19.117,0.042,2.265



=== TEXT STATISTICS ===

Column: pidgin_transcription
Total words: 659853
Unique words: 21055
Average words: 64.58
Median words: 61.0
Min words: 1
Max words: 233
Average characters: 339.34
Missing: 0

Column: english_translation
Total words: 660891
Unique words: 23711
Average words: 64.68
Median words: 62.0
Min words: 1
Max words: 225
Average characters: 357.47
Missing: 0

=== MEDICAL TERM STATISTICS ===

Column: medical_terms_english
Total term occurrences: 64288
Unique medical terms: 22606
Utterances containing terms: 9887
Average terms per utterance: 6.29
Top 20 medical terms:


,Medical Term,Frequency
0,pain,1278
1,normal,304
2,ultrasound,287
3,antibiotics,283
4,symptoms,278
5,swelling,270
6,back,270
7,medications,244
8,medication,229
9,fever,226



Column: medical_terms_pidgen
Total term occurrences: 68185
Unique medical terms: 21888
Utterances containing terms: 9935
Average terms per utterance: 6.67
Top 20 medical terms:


...

,count,mean,std,min,25%,50%,75%,max
audio_length,10218.0,27.826,12.653,2.150,19.310,25.310,34.025,124.390
sample_rate,10218.0,16000.000,0.000,16000.000,16000.000,16000.000,16000.000,16000.000
num_channels,10218.0,1.000,0.000,1.000,1.000,1.000,1.000,1.000
speech_rate,10218.0,2.363,0.492,0.132,2.029,2.347,2.686,14.695
signal_to_noise_ratio,10218.0,32.039,10.505,5.822,25.183,30.804,37.089,65.283
silence_ratio,10218.0,0.188,0.112,0.000,0.120,0.187,0.259,0.972



Total Audio Duration: 78h 58m 49s
Total Hours: 78.98

Sample Rate Distribution:


,count
sample_rate,
16000.0,10218



Channel Distribution:


,count
num_channels,
1.0,10218


In [21]:
summary = data[numeric_cols].agg([
    "count",
    "mean",
    "std",
    "min",
    "median",
    "max"
]).T.round(3)

display(summary)

,count,mean,std,min,median,max
audio_length,10218.0,27.826,12.653,2.150,25.310,124.390
sample_rate,10218.0,16000.000,0.000,16000.000,16000.000,16000.000
num_channels,10218.0,1.000,0.000,1.000,1.000,1.000
speech_rate,10218.0,2.363,0.492,0.132,2.347,14.695
signal_to_noise_ratio,10218.0,32.039,10.505,5.822,30.804,65.283
silence_ratio,10218.0,0.188,0.112,0.000,0.187,0.972


In [22]:
summary = data[numeric_cols].describe().T.round(3)
display(summary)

,count,mean,std,min,25%,50%,75%,max
audio_length,10218.0,27.826,12.653,2.150,19.310,25.310,34.025,124.390
sample_rate,10218.0,16000.000,0.000,16000.000,16000.000,16000.000,16000.000,16000.000
num_channels,10218.0,1.000,0.000,1.000,1.000,1.000,1.000,1.000
speech_rate,10218.0,2.363,0.492,0.132,2.029,2.347,2.686,14.695
signal_to_noise_ratio,10218.0,32.039,10.505,5.822,25.183,30.804,37.089,65.283
silence_ratio,10218.0,0.188,0.112,0.000,0.120,0.187,0.259,0.972
